# Predict Customer Churn - Kaggle GPU Notebook (XGBoost)

This notebook contains the complete pipeline for the Kaggle Playground Series competition **"Predict Customer Churn"**. It combines data loading, preprocessing, hyperparameter tuning with Optuna, and cross-validated training using **XGBoost**.

**Hardware Switch:** This notebook includes conditions to detect a Kaggle environment and adjust paths and GPU parameters accordingly. Ensure the accelerator is set to **GPU (T4x2 or P100)** in the Kaggle notebook settings.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
import warnings
import os
import gc

warnings.filterwarnings('ignore')

# --- 1. SETTINGS & PATHS ---
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle environment.")
    # Note: Replace 'playground-series-s6e3' with the exact dataset name if it differs on Kaggle
    DATA_DIR = '/kaggle/input/playground-series-s6e3/'
else:
    print("Running locally.")
    DATA_DIR = '../data/'  # Assuming notebook is in 'notebooks/' and data is in 'data/'

# Flag to run Optuna tuning or just use pre-computed best params
RUN_TUNING = True
N_TRIALS = 20
N_SPLITS = 5

## 2. Load & Preprocess Data

In [ ]:
print(f"Loading data from {DATA_DIR}...")
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

# Target variable formatting
target = 'Churn'
if train[target].dtype == 'object':
    train[target] = train[target].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# Combine for consistent preprocessing
train['is_train'] = 1
test['is_train'] = 0
df = pd.concat([train, test], ignore_index=True)

features = [c for c in train.columns if c not in ['id', target, 'is_train']]
categorical_features = []

# Encode categorical features
for col in features:
    if df[col].dtype == 'object':
        categorical_features.append(col)
        le = LabelEncoder()
        df[col] = df[col].fillna('Missing')
        df[col] = le.fit_transform(df[col].astype(str))

# Fill missing numerical features
num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Split back into train/test
train_df = df[df['is_train'] == 1].reset_index(drop=True)
test_df = df[df['is_train'] == 0].reset_index(drop=True)

X = train_df[features]
y = train_df[target]
X_test = test_df[features]

print(f"Train shape: {X.shape}, Test shape: {X_test.shape}")

## 3. Hyperparameter Tuning (Optuna)

In [ ]:
best_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42,
}

# Add GPU acceleration if requested in Kaggle
if IS_KAGGLE:
    best_params['tree_method'] = 'hist' # modern replacement for gpu_hist
    best_params['device'] = 'cuda'

if RUN_TUNING:
    print("Starting Optuna Hyperparameter Tuning over a Validation Split...")
    X_t, X_v, y_t, y_v = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # Note: Enable categorical support natively for XGBoost
    X_t = X_t.copy()
    X_v = X_v.copy()
    for col in categorical_features:
        X_t[col] = X_t[col].astype('category')
        X_v[col] = X_v[col].astype('category')
    
    dtrain = xgb.DMatrix(X_t, label=y_t, enable_categorical=True)
    dvalid = xgb.DMatrix(X_v, label=y_v, enable_categorical=True)

    def objective(trial):
        params = best_params.copy()
        params.update({
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
            'alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
            'lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
        })
        
        # Early stopping is passed via the early_stopping_rounds parameter to xgb.train in newer versions
        # The evaluation metric must be the LAST item in evals for early stopping to work correctly.
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=1500,
            evals=[(dtrain, 'train'), (dvalid, 'valid')],
            early_stopping_rounds=50,
            verbose_eval=False
        )
        
        # model.best_iteration points to the best iteration if early_stopping_rounds was used
        valid_preds = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
        score = roc_auc_score(y_v, valid_preds)
        return score

    # Suppress XGBoost warnings during tuning 
    xgb.set_config(verbosity=0)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    
    print("\n--- OPTUNA TUNING COMPLETE ---")
    print(f"Best AUC: {study.best_value:.5f}")
    print("Best Params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
        best_params[k] = v
        
    # Revert verbosity
    xgb.set_config(verbosity=1)
else:
    print("Using predefined Best Parameters...")
    # Precomputed params from a previous XGBoost run
    precomputed_params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'min_child_weight': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    }
    best_params.update(precomputed_params)

## 4. Final Training & Stratified K-Fold

In [ ]:
print("\nStarting Stratified K-Fold Training...")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
scores = []

X_xgb = X.copy()
X_test_xgb = X_test.copy()

# Must explicitly cast string encoded categories to Pandas 'category' dtype for XGBoost natively
for col in categorical_features:
    X_xgb[col] = X_xgb[col].astype('category')
    X_test_xgb[col] = X_test_xgb[col].astype('category')

dtest = xgb.DMatrix(X_test_xgb, enable_categorical=True)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_xgb, y)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    X_train_f, y_train_f = X_xgb.iloc[train_idx], y.iloc[train_idx]
    X_valid_f, y_valid_f = X_xgb.iloc[valid_idx], y.iloc[valid_idx]
    
    dtrain = xgb.DMatrix(X_train_f, label=y_train_f, enable_categorical=True)
    dvalid = xgb.DMatrix(X_valid_f, label=y_valid_f, enable_categorical=True)
    
    model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=2000,
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        early_stopping_rounds=150,
        verbose_eval=200
    )
    
    valid_preds = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
    oof_preds[valid_idx] = valid_preds
    
    # Accumulate test predictions across folds
    test_preds += model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) / N_SPLITS
    
    score = roc_auc_score(y_valid_f, valid_preds)
    scores.append(score)
    print(f"Fold {fold+1} AUC: {score:.5f}")

print("\n===========================")
print(f"Mean Fold AUC:   {np.mean(scores):.5f}")
print(f"Overall OOF AUC: {roc_auc_score(y, oof_preds):.5f}")
print("===========================")

## 5. Prepare Submission

In [ ]:
sub[target] = test_preds
sub.to_csv('submission_xgboost.csv', index=False)
print("\nSubmission saved to 'submission_xgboost.csv'!")
display(sub.head())